# 04 — Évaluation & Inférence

Test du pipeline complet d'inférence sur nouvelles données.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid")
FAKE_DATA_PATH = "../../data/raw/fake_mesures.json"


## 1. Chargement des modèles & prédiction

In [ ]:
from ml.src.data.loader import load_combined
from ml.src.data.cleaner import clean
from ml.src.data.feature_builder import build_features
from ml.src.pipelines.inference_pipeline import load_models, predict

df = load_combined(fake_path=FAKE_DATA_PATH, prefer_api=False)
df = clean(df)
df = build_features(df)

# Prendre les 20% derniers comme "données de production"
test_df = df.tail(int(len(df) * 0.2)).copy()
print(f"Données test : {len(test_df)} lignes")

clf, reg, scaler, le, scaler_reg = load_models()
test_df = predict(test_df, clf, reg, scaler, le, scaler_reg)
test_df[["timestamp", "humidite_sol", "pred_etat_sol", "pred_besoin_eau", "pred_humidite"]].head(10)


## 2. Prédictions vs Réalité — Régresseur

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(test_df["timestamp"], test_df["humidite_prevue"],
         label="Humidité réelle (future)", color="#4e79a7", lw=1.5)
plt.plot(test_df["timestamp"], test_df["pred_humidite"],
         label="Humidité prédite", color="#e15759", lw=1.5, linestyle="--")
plt.fill_between(test_df["timestamp"],
                 test_df["humidite_prevue"], test_df["pred_humidite"],
                 alpha=0.15, color="gray")
plt.xlabel("Timestamp")
plt.ylabel("Humidité Sol (%)")
plt.title("Prédictions Régresseur vs Réalité", fontweight="bold")
plt.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig("../../ml/reports/figures/08_regressor_predictions.png", bbox_inches="tight")
plt.show()


## 3. Dry-run du pipeline d'inférence complet

In [ ]:
# Lance le pipeline sans envoyer à l'API (dry_run=True)
from ml.src.pipelines.inference_pipeline import run
df_result = run(limit=50, dry_run=True)
